# Music Genre Classification Capstone Project

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
import librosa

In [ ]:
USE_3_SEC = True

## Data

In [4]:
data = pd.read_csv('data/features_3_sec.csv') if USE_3_SEC else pd.read_csv('data/features_30_sec.csv')

In [5]:
le = LabelEncoder()

X = data.drop(columns=['filename', 'length', 'label'])
y = le.fit_transform(data['label'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [59]:
data['label'].value_counts()

label
blues        1000
jazz         1000
metal        1000
pop          1000
reggae       1000
disco         999
classical     998
hiphop        998
rock          998
country       997
Name: count, dtype: int64

In [6]:
print(len(y_test))

1998


## Model

### Which model performs the best?

In [ ]:
# Create a placeholder pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC()) # Placeholder
])

# Define a list of dictionaries for different models
# Test KNN, LogReg
# Test other metrics (f1 score)

param_grid = [
    {
        'clf': [SVC()],
        'clf__kernel': ['linear', 'rbf'],
        'clf__C': [0.1, 1, 10]
    },
    {
        'clf': [RandomForestClassifier()],
        'clf__n_estimators': [50, 100, 200],
        'clf__max_depth': [None, 10, 20]
    }
]

grid = GridSearchCV(pipe, param_grid, cv=5, refit=True)
grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...clf', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'clf': [SVC()], 'clf__C': [0.1, 1, ...], 'clf__kernel': ['linear', 'rbf']}, {'clf': [RandomForestClassifier()], 'clf__max_depth': [None, 10, ...], 'clf__n_estimators': [50, 100, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the mo

In [34]:
print(f'Best Model:{grid.best_params_}')
print(f"Accuracy: {grid.score(X_test, y_test):.2f}")

## Model Evaluation
y_pred = grid.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Best Model:{'clf': SVC(), 'clf__C': 10, 'clf__kernel': 'rbf'}
Accuracy: 0.93
              precision    recall  f1-score   support

           0       0.92      0.97      0.94       208
           1       0.90      0.99      0.94       203
           2       0.89      0.90      0.90       186
           3       0.92      0.91      0.92       199
           4       0.94      0.94      0.94       218
           5       0.94      0.92      0.93       192
           6       0.97      0.99      0.98       204
           7       0.93      0.95      0.94       180
           8       0.95      0.93      0.94       211
           9       0.93      0.82      0.87       197

    accuracy                           0.93      1998
   macro avg       0.93      0.93      0.93      1998
weighted avg       0.93      0.93      0.93      1998

[[201   0   2   1   0   2   0   0   1   1]
 [  1 200   0   0   0   2   0   0   0   0]
 [  6   2 168   1   1   3   0   3   0   2]
 [  1   3   2 182   2   0   0   3  

### What are the best hyperparameters?

In [52]:
svc_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),
    ('svc', SVC(kernel='rbf'))
])

svc_param_grid = {
    'svc__C': [10, 15, 25],
}

svc_grid = GridSearchCV(svc_pipe, svc_param_grid, cv=5, refit=True, verbose=2)
svc_grid.fit(X_train, y_train)

Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV] END ..........................................svc__C=10; total time=   0.6s
[CV] END ..........................................svc__C=10; total time=   0.6s
[CV] END ..........................................svc__C=10; total time=   0.6s
[CV] END ..........................................svc__C=10; total time=   0.6s
[CV] END ..........................................svc__C=10; total time=   0.6s
[CV] END ..........................................svc__C=15; total time=   0.6s
[CV] END ..........................................svc__C=15; total time=   0.6s
[CV] END ..........................................svc__C=15; total time=   0.6s
[CV] END ..........................................svc__C=15; total time=   0.6s
[CV] END ..........................................svc__C=15; total time=   0.6s
[CV] END ..........................................svc__C=25; total time=   0.6s
[CV] END ........................................

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...svc', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'svc__C': [10, 15, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate param

In [54]:
print(f'Best C: {svc_grid.best_params_['svc__C']}')
print(f'Accuracy: {svc_grid.score(X_test, y_test):.2f}')

svc_pred = svc_grid.predict(X_test)
print(classification_report(y_test, svc_pred))
print(confusion_matrix(y_test, svc_pred))

Best C: 25
Accuracy: 0.91
              precision    recall  f1-score   support

           0       0.90      0.93      0.91       208
           1       0.88      0.97      0.92       203
           2       0.86      0.87      0.87       186
           3       0.90      0.89      0.90       199
           4       0.94      0.92      0.93       218
           5       0.91      0.88      0.90       192
           6       0.96      0.96      0.96       204
           7       0.88      0.96      0.91       180
           8       0.93      0.91      0.92       211
           9       0.90      0.77      0.83       197

    accuracy                           0.91      1998
   macro avg       0.91      0.91      0.91      1998
weighted avg       0.91      0.91      0.91      1998

[[193   1   5   3   0   3   0   0   1   2]
 [  1 197   0   0   0   5   0   0   0   0]
 [  9   2 162   3   0   3   0   5   0   2]
 [  3   3   1 178   2   1   0   5   3   3]
 [  2   2   4   1 201   0   0   4   3   1]


## Test on New Audio File

In [48]:
def extract_features(file_path):
    # 1. Load the audio file
    # GTZAN standard is sr=22050. Duration is usually 30s for the main dataset.
    y, sr = librosa.load(file_path, mono=True, duration=30)
    
    # 2. Compute Features
    # Chroma STFT
    chroma_stft = librosa.feature.chroma_stft(y=y, sr=sr)
    # RMS
    rms = librosa.feature.rms(y=y)
    # Spectral Centroid
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    # Spectral Bandwidth
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    # Rolloff
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y)
    # Harmony and Percussive
    harmony, perceptr = librosa.effects.hpss(y)
    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    # MFCCs (20 coefficients)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)

    # 3. Aggregate Features (Mean and Variance)
    # The dataframe must match the order and naming convention of GTZAN
    features = {
        'chroma_stft_mean': np.mean(chroma_stft),
        'chroma_stft_var': np.var(chroma_stft),
        'rms_mean': np.mean(rms),
        'rms_var': np.var(rms),
        'spectral_centroid_mean': np.mean(spec_cent),
        'spectral_centroid_var': np.var(spec_cent),
        'spectral_bandwidth_mean': np.mean(spec_bw),
        'spectral_bandwidth_var': np.var(spec_bw),
        'rolloff_mean': np.mean(rolloff),
        'rolloff_var': np.var(rolloff),
        'zero_crossing_rate_mean': np.mean(zcr),
        'zero_crossing_rate_var': np.var(zcr),
        'harmony_mean': np.mean(harmony),
        'harmony_var': np.var(harmony),
        'perceptr_mean': np.mean(perceptr),
        'perceptr_var': np.var(perceptr),
        'tempo': float(np.atleast_1d(tempo).flat[0]),
    }

    # Add MFCCs 1-20 (Mean and Variance)
    for i in range(1, 21):
        features[f'mfcc{i}_mean'] = np.mean(mfccs[i-1])
        features[f'mfcc{i}_var'] = np.var(mfccs[i-1])

    return features

# Example Usage:
# feature_dict = extract_features('my_song.wav')
# df_inference = pd.DataFrame([feature_dict])

In [ ]:
feature_dict = extract_features('data/test_songs/Tennessee_Whiskey.wav')
df_inference = pd.DataFrame([feature_dict])

,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,rolloff_mean,rolloff_var,...,mfcc16_mean,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var
0,0.39098,0.101822,0.087732,0.001552,1640.383112,744574.448308,2157.661436,340810.490326,3535.836953,4.257630e+06,...,-3.820747,49.176731,-4.375934,39.383236,-0.094272,57.000908,-2.488606,50.494678,-0.386924,48.492458


In [55]:
# Generate prediction and convert label encoded prediction back to genre
prediction = svc_grid.predict(df_inference)
prediction_genre = le.inverse_transform(prediction)
print(f'Prediction: {prediction_genre[0]}')


Prediction: reggae
